# Моя часть проекта: анализ данных, KPI и прогнозирование продаж

**Студент:** Азаб А. М.  
**Группа:** РИ-420947  
**Роль в проекте:** ML-разработчик + Data Scientist  
**Тема ВКР:** «Разработка модуля анализа маркетинговых данных с автоматической подготовкой, расчетом KPI и прогнозированием продаж»

---

Этот ноутбук показывает конкретно мою часть работы над проектом — от первичного разбора данных до оценки качества финальной модели. Всё, что здесь показано, основано на реальных файлах репозитория, реальных артефактах обучения и реальных числах из `ml/artifacts/model_metadata.json`.

Сам проект — это аналитическая платформа для прогнозирования розничных продаж (датасет Rossmann). Моя задача — взять сырые CSV-данные, разобраться в их структуре, подготовить признаки для модели, обучить несколько алгоритмов и проверить, как хорошо они предсказывают. Плюс — убедиться, что то, что я сделал, корректно подключается к бэкенду платформы.

Почему это важно: без нормальной подготовки данных никакой алгоритм не поможет. Это я понял довольно рано, когда первые попытки предсказания давали огромные ошибки именно из-за плохо обработанных пропусков и не учтённой сезонности.

## 1. Ориентация в репозитории

Перед тем как разбирать данные, я проверяю, что все нужные файлы на месте. Это хорошая привычка — сначала убедиться в структуре, потом лезть в код.

In [1]:
import os
import sys
from pathlib import Path

# Работаем из корня репозитория
ROOT = Path('.').resolve()
# Если запускаем из папки notebooks/, поднимаемся на уровень выше
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
    os.chdir(ROOT)

print(f'Корень проекта: {ROOT}')
print()

# Файлы, которые важны для моей роли
key_files = {
    'data/train.csv':                          'Исторические продажи по магазинам',
    'data/store.csv':                          'Метаданные магазинов',
    'ml/features.py':                          'Функции формирования признаков',
    'ml/train.py':                             'Скрипт обучения моделей',
    'ml/config.yaml':                          'Конфигурация: гиперпараметры и пути',
    'ml/artifacts/model_metadata.json':        'Метаданные финальной модели (метрики)',
    'ml/artifacts/model.joblib':               'Сохранённый артефакт модели',
    'sql/01_schema.sql':                       'DDL звёздной схемы DWH',
    'sql/02_views_kpi.sql':                    'KPI-представления поверх DWH',
    'backend/app/services/forecast_service.py':'Сервис инференса модели в бэкенде',
    'backend/app/routers/forecast.py':         'REST-эндпоинты прогноза',
}

print(f'{"Файл":<48} {"Есть?":<8} Роль')
print('-' * 100)
for fpath, desc in key_files.items():
    exists = '✓' if (ROOT / fpath).exists() else '✗ MISSING'
    print(f'{fpath:<48} {exists:<8} {desc}')

Корень проекта: /home/azab-22/Desktop/sales-forecasting-analytics-platform-rossmann

Файл                                             Есть?    Роль
----------------------------------------------------------------------------------------------------
data/train.csv                                   ✓        Исторические продажи по магазинам
data/store.csv                                   ✓        Метаданные магазинов
ml/features.py                                   ✓        Функции формирования признаков
ml/train.py                                      ✓        Скрипт обучения моделей
ml/config.yaml                                   ✓        Конфигурация: гиперпараметры и пути
ml/artifacts/model_metadata.json                 ✓        Метаданные финальной модели (метрики)
ml/artifacts/model.joblib                        ✓        Сохранённый артефакт модели
sql/01_schema.sql                                ✓        DDL звёздной схемы DWH
sql/02_views_kpi.sql                             ✓  

## 2. Загрузка и первичный осмотр данных

Датасет Rossmann Store Sales — это исторические данные о ежедневных продажах сети аптек в Германии. Два основных файла:

- `train.csv` — факт: когда, в каком магазине, сколько продали, сколько покупателей, было ли промо;
- `store.csv` — справочник магазинов: тип, ассортимент, расстояние до ближайшего конкурента.

Начинаю с того, что просто смотрю на данные, не торопясь делать выводы.

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Загрузка
train = pd.read_csv('data/train.csv', parse_dates=['Date'], dtype={'StateHoliday': str})
store = pd.read_csv('data/store.csv')

# Нижний регистр для удобства
train.columns = train.columns.str.lower()
store.columns = store.columns.str.lower()

print(f'train.csv: {train.shape[0]:,} строк × {train.shape[1]} столбцов')
print(f'store.csv: {store.shape[0]:,} строк × {store.shape[1]} столбцов')
print()
print('Столбцы train:', list(train.columns))
print('Столбцы store:', list(store.columns))

train.csv: 1,017,209 строк × 9 столбцов
store.csv: 1,115 строк × 10 столбцов

Столбцы train: ['store', 'dayofweek', 'date', 'sales', 'customers', 'open', 'promo', 'stateholiday', 'schoolholiday']
Столбцы store: ['store', 'storetype', 'assortment', 'competitiondistance', 'competitionopensincemonth', 'competitionopensinceyear', 'promo2', 'promo2sinceweek', 'promo2sinceyear', 'promointerval']


In [3]:
print('Первые строки train.csv:')
display(train.head(5))

print('\nПервые строки store.csv:')
display(store.head(5))

Первые строки train.csv:


,store,dayofweek,date,sales,customers,open,promo,stateholiday,schoolholiday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1



Первые строки store.csv:


,store,storetype,assortment,competitiondistance,competitionopensincemonth,competitionopensinceyear,promo2,promo2sinceweek,promo2sinceyear,promointerval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [4]:
print('Типы данных train:')
print(train.dtypes)
print()
print('Базовая статистика по sales и customers:')
display(train[['sales','customers','open','promo']].describe().round(2))

Типы данных train:
store                     int64
dayofweek                 int64
date             datetime64[ns]
sales                     int64
customers                 int64
open                      int64
promo                     int64
stateholiday             object
schoolholiday             int64
dtype: object

Базовая статистика по sales и customers:


,sales,customers,open,promo
count,1017209.00,1017209.00,1017209.00,1017209.00
mean,5773.82,633.15,0.83,0.38
std,3849.93,464.41,0.38,0.49
min,0.00,0.00,0.00,0.00
25%,3727.00,405.00,1.00,0.00
50%,5744.00,609.00,1.00,0.00
75%,7856.00,837.00,1.00,1.00
max,41551.00,7388.00,1.00,1.00


## 3. Проверка качества данных

Прежде чем строить признаки, нужно понять, с какими проблемами придётся разбираться. В реальных ритейл-датасетах почти всегда есть пропуски, нули в продажах в дни закрытия магазина, выбросы. Если это не учесть — модель будет учиться на мусоре.

In [5]:
print('=== Пропуски в train.csv ===')
missing_train = train.isnull().sum()
print(missing_train[missing_train > 0] if missing_train.any() else 'Пропусков нет')

print('\n=== Пропуски в store.csv ===')
missing_store = store.isnull().sum()
print(missing_store[missing_store > 0])

=== Пропуски в train.csv ===
Пропусков нет

=== Пропуски в store.csv ===
competitiondistance            3
competitionopensincemonth    354
competitionopensinceyear     354
promo2sinceweek              544
promo2sinceyear              544
promointerval                544
dtype: int64


In [6]:
# Временной диапазон
print(f"Временной диапазон данных: {train['date'].min().date()} → {train['date'].max().date()}")
print(f"Всего уникальных дат: {train['date'].nunique()}")
print(f"Всего уникальных магазинов: {train['store'].nunique()}")

print()
# Строки с нулевыми продажами — это закрытые магазины
zero_sales = (train['sales'] == 0).sum()
closed = (train['open'] == 0).sum()
print(f"Строк с sales=0: {zero_sales:,} ({zero_sales/len(train)*100:.1f}%)")
print(f"Строк с open=0:  {closed:,} ({closed/len(train)*100:.1f}%)")
print()
print('Важно: строки с open=0 нужно убирать при обучении, иначе модель')
print('будет учиться предсказывать нулевые продажи в выходные — это неверно.')

Временной диапазон данных: 2013-01-01 → 2015-07-31
Всего уникальных дат: 942
Всего уникальных магазинов: 1115

Строк с sales=0: 172,871 (17.0%)
Строк с open=0:  172,817 (17.0%)

Важно: строки с open=0 нужно убирать при обучении, иначе модель
будет учиться предсказывать нулевые продажи в выходные — это неверно.


In [7]:
print('Распределение StateHoliday:')
print(train['stateholiday'].value_counts())
print()
print('Распределение SchoolHoliday:')
print(train['schoolholiday'].value_counts())
print()
print('Промо-дни vs обычные дни:')
print(train['promo'].value_counts())
print()
print('Типы магазинов (store.csv):')
print(store['storetype'].value_counts())
print()
print('Ассортимент (store.csv):')
print(store['assortment'].value_counts())

Распределение StateHoliday:


stateholiday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64

Распределение SchoolHoliday:
schoolholiday
0    835488
1    181721
Name: count, dtype: int64

Промо-дни vs обычные дни:


promo
0    629129
1    388080
Name: count, dtype: int64

Типы магазинов (store.csv):
storetype
a    602
d    348
c    148
b     17
Name: count, dtype: int64

Ассортимент (store.csv):
assortment
a    593
c    513
b      9
Name: count, dtype: int64


In [8]:
# Пропуски в CompetitionDistance — это реальная проблема
comp_na = store['competitiondistance'].isnull().sum()
print(f'Пропуски в CompetitionDistance: {comp_na} из {len(store)}')
print('Решение в проекте: заполняем нулём (COALESCE(competition_distance, 0) в SQL),')
print('плюс добавляем log1p-преобразование, чтобы сгладить правый скос распределения.')
print()
print('Статистика CompetitionDistance (без NA):')
print(store['competitiondistance'].describe().round(0))

Пропуски в CompetitionDistance: 3 из 1115
Решение в проекте: заполняем нулём (COALESCE(competition_distance, 0) в SQL),
плюс добавляем log1p-преобразование, чтобы сгладить правый скос распределения.

Статистика CompetitionDistance (без NA):
count     1112.0
mean      5405.0
std       7663.0
min         20.0
25%        718.0
50%       2325.0
75%       6882.0
max      75860.0
Name: competitiondistance, dtype: float64


## 4. KPI-анализ маркетинговых данных

Одна из задач моей части — не просто обучить модель, но и рассчитать базовые KPI, которые потом отображаются в аналитической панели. Здесь я смотрю на агрегаты, которые соответствуют тому, что выдают SQL-вьюшки (`v_kpi_summary`, `v_promo_impact`, `v_top_stores_by_sales`) в продакшн-базе.

In [9]:
# Работаем только с открытыми магазинами
df_open = train[train['open'] == 1].copy()
df_open['date'] = pd.to_datetime(df_open['date'])
df_open['year'] = df_open['date'].dt.year

print('=== Базовые KPI по всему датасету ===')
total_sales      = df_open['sales'].sum()
total_customers  = df_open['customers'].sum()
avg_daily_sales  = df_open.groupby('date')['sales'].sum().mean()
total_promo_days = (df_open['promo'] == 1).sum()

print(f'  Всего продаж (сумма):        {total_sales:>20,.0f} €')
print(f'  Всего покупателей:           {total_customers:>20,.0f}')
print(f'  Ср. продажи за день (все):   {avg_daily_sales:>20,.1f} €')
print(f'  Дней с промо-акцией:         {total_promo_days:>20,.0f}')

=== Базовые KPI по всему датасету ===
  Всего продаж (сумма):               5,873,180,623 €
  Всего покупателей:                    644,041,755
  Ср. продажи за день (все):            6,234,799.0 €
  Дней с промо-акцией:                      376,896


In [10]:
# Промо-эффект — ключевая метрика для маркетинга
promo_stats = df_open.groupby('promo')['sales'].agg(['mean', 'count'])
promo_stats.index = ['Без промо', 'С промо']
promo_stats.columns = ['Средние продажи, €', 'Кол-во дней']
promo_stats = promo_stats.round(2)
print('=== Промо vs Без промо ===')
display(promo_stats)

avg_promo    = df_open[df_open['promo']==1]['sales'].mean()
avg_no_promo = df_open[df_open['promo']==0]['sales'].mean()
uplift = (avg_promo - avg_no_promo) / avg_no_promo * 100
print(f'\nPromo uplift: +{uplift:.1f}%  — промо-дни дают примерно на {uplift:.0f}% выше продажи')

=== Промо vs Без промо ===


,"Средние продажи, €",Кол-во дней
Без промо,5929.41,467496
С промо,8228.28,376896



Promo uplift: +38.8%  — промо-дни дают примерно на 39% выше продажи


In [11]:
# Топ-10 магазинов по суммарным продажам
top_stores = (
    df_open.groupby('store')['sales']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_stores.columns = ['store_id', 'total_sales']
top_stores['total_sales'] = top_stores['total_sales'].map('{:,.0f}'.format)
print('=== Топ-10 магазинов по суммарным продажам ===')
display(top_stores)

=== Топ-10 магазинов по суммарным продажам ===


,store_id,total_sales
0,262,"19,516,842"
1,817,"17,057,867"
2,562,"16,927,322"
3,1114,"16,202,585"
4,251,"14,896,870"
5,513,"14,252,406"
6,788,"14,082,141"
7,733,"14,067,158"
8,383,"13,489,879"
9,756,"12,911,782"


In [12]:
# Продажи по типам магазинов
df_merged = df_open.merge(store[['store','storetype','assortment']], on='store', how='left')
by_type = df_merged.groupby('storetype')['sales'].agg(['mean','sum','count'])
by_type.columns = ['Ср. продажи/день, €', 'Суммарные продажи, €', 'Кол-во записей']
by_type = by_type.round(0)
print('=== KPI по типам магазинов ===')
display(by_type)

=== KPI по типам магазинов ===


,"Ср. продажи/день, €","Суммарные продажи, €",Кол-во записей
storetype,,,
a,6925.0,3165334859,457077
b,10231.0,159231395,15563
c,6933.0,783221426,112978
d,6822.0,1765392943,258774


### Графики: визуализация ключевых паттернов

Без визуализации тяжело объяснить бизнесу, что происходит в данных. Здесь я смотрю на сезонность и промо-эффект — две главные движущие силы, которые модель должна уметь улавливать.

In [13]:
import matplotlib
matplotlib.use('Agg')  # без GUI
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('KPI и паттерны данных Rossmann', fontsize=14, fontweight='bold', y=1.01)

# --- 1. Суммарные продажи по месяцам ---
monthly = df_open.copy()
monthly['month'] = monthly['date'].dt.to_period('M').dt.to_timestamp()
monthly_sales = monthly.groupby('month')['sales'].sum() / 1e6

ax = axes[0, 0]
ax.plot(monthly_sales.index, monthly_sales.values, color='#0a7e64', linewidth=1.8)
ax.fill_between(monthly_sales.index, monthly_sales.values, alpha=0.15, color='#0a7e64')
ax.set_title('Суммарные продажи по месяцам, млн €')
ax.set_xlabel('Месяц')
ax.set_ylabel('Продажи, млн €')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.tick_params(axis='x', rotation=30)
ax.grid(True, alpha=0.3)

# --- 2. Средние продажи по дням недели ---
dow_map = {1:'Пн',2:'Вт',3:'Ср',4:'Чт',5:'Пт',6:'Сб',7:'Вс'}
by_dow = df_open.groupby('dayofweek')['sales'].mean()

ax = axes[0, 1]
colors = ['#0a7e64' if i != 7 else '#e55' for i in by_dow.index]
ax.bar([dow_map[i] for i in by_dow.index], by_dow.values, color=colors, edgecolor='white')
ax.set_title('Ср. продажи по дням недели')
ax.set_ylabel('Продажи, €')
ax.grid(axis='y', alpha=0.3)

# --- 3. Промо-эффект ---
ax = axes[1, 0]
promo_vals  = [avg_no_promo, avg_promo]
promo_lbls  = [f'Без промо\n{avg_no_promo:,.0f} €', f'С промо\n{avg_promo:,.0f} €']
bar_colors  = ['#aaa', '#0a7e64']
bars = ax.bar(promo_lbls, promo_vals, color=bar_colors, width=0.4, edgecolor='white')
ax.set_title(f'Промо-эффект: +{uplift:.1f}% к продажам')
ax.set_ylabel('Ср. продажи, €')
ax.grid(axis='y', alpha=0.3)
ax.bar_label(bars, fmt='%.0f €', padding=4)

# --- 4. Распределение продаж (без нулей) ---
ax = axes[1, 1]
sales_nonzero = df_open[df_open['sales'] > 0]['sales']
ax.hist(sales_nonzero, bins=80, color='#0a7e64', edgecolor='none', alpha=0.8)
ax.axvline(sales_nonzero.mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Среднее: {sales_nonzero.mean():,.0f}')
ax.axvline(sales_nonzero.median(), color='orange', linestyle='--', linewidth=1.5,
           label=f'Медиана: {sales_nonzero.median():,.0f}')
ax.set_title('Распределение продаж (ненулевые)')
ax.set_xlabel('Продажи, €')
ax.set_ylabel('Частота')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('notebooks/kpi_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('График сохранён: notebooks/kpi_overview.png')

График сохранён: notebooks/kpi_overview.png


**Что видно на графиках:**

1. В продажах есть явная сезонность — декабрь стабильно выше, летом наблюдается некоторый спад. Это модель должна уловить через лаговые признаки с окном 364 дня.

2. Воскресенье отдаёт заметно меньше продаж (или нулей, если магазин закрыт). Это важный признак `day_of_week`.

3. Промо даёт ощутимый прирост — это подтверждает, что `promo` и `promo_density` будут полезными признаками.

4. Распределение продаж правоскошенное — отсюда решение применить `log1p`-преобразование целевой переменной перед обучением.

## 5. Связь с DWH: звёздная схема

Сырые CSV-файлы — это только входная точка. В продакшн-варианте платформы данные загружаются в PostgreSQL со звёздной схемой. Я хочу показать, как именно CSV-поля маппятся на таблицы DWH — это важно понимать, потому что сервис прогнозирования тянет данные уже из базы, не из CSV.

In [14]:
# Читаем схему из файлов репозитория
with open('sql/01_schema.sql', encoding='utf-8') as f:
    schema_sql = f.read()

with open('sql/02_views_kpi.sql', encoding='utf-8') as f:
    views_sql = f.read()

print('=== sql/01_schema.sql — первые 40 строк ===')
print('\n'.join(schema_sql.splitlines()[:40]))

=== sql/01_schema.sql — первые 40 строк ===
﻿-- 01_schema.sql
-- Star Schema for Rossmann daily sales forecasting

CREATE TABLE IF NOT EXISTS dim_store (
    store_id INTEGER PRIMARY KEY,
    store_type VARCHAR(16),
    assortment VARCHAR(16),
    competition_distance NUMERIC(12, 2),
    competition_open_since_month INTEGER,
    competition_open_since_year INTEGER,
    promo2 INTEGER,
    promo2_since_week INTEGER,
    promo2_since_year INTEGER,
    promo_interval VARCHAR(64)
);

CREATE TABLE IF NOT EXISTS dim_date (
    date_id BIGSERIAL PRIMARY KEY,
    full_date DATE NOT NULL UNIQUE,
    day INTEGER NOT NULL,
    month INTEGER NOT NULL,
    year INTEGER NOT NULL,
    quarter INTEGER NOT NULL,
    week_of_year INTEGER NOT NULL,
    day_of_week INTEGER NOT NULL,
    is_weekend BOOLEAN NOT NULL
);

CREATE TABLE IF NOT EXISTS fact_sales_daily (
    id BIGSERIAL PRIMARY KEY,
    store_id INTEGER NOT NULL,
    date_id BIGINT NOT NULL,
    sales NUMERIC(14, 2) NOT NULL,
    customers INTEG

In [15]:
print('=== sql/02_views_kpi.sql — первые 35 строк ===')
print('\n'.join(views_sql.splitlines()[:35]))

=== sql/02_views_kpi.sql — первые 35 строк ===
﻿-- 02_views_kpi.sql

CREATE OR REPLACE VIEW v_kpi_summary AS
SELECT
    d.full_date,
    f.store_id,
    SUM(f.sales) AS total_sales,
    SUM(COALESCE(f.customers, 0)) AS total_customers,
    AVG(f.sales) AS avg_sales,
    SUM(CASE WHEN COALESCE(f.promo, 0) = 1 THEN 1 ELSE 0 END) AS promo_days,
    SUM(CASE WHEN COALESCE(f.open, 1) = 1 THEN 1 ELSE 0 END) AS open_days
FROM fact_sales_daily f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.full_date, f.store_id;

CREATE OR REPLACE VIEW v_sales_timeseries_daily AS
SELECT
    d.full_date,
    f.store_id,
    SUM(f.sales) AS sales,
    SUM(COALESCE(f.customers, 0)) AS customers,
    MAX(COALESCE(f.promo, 0)) AS promo,
    MAX(COALESCE(f.open, 1)) AS open
FROM fact_sales_daily f
JOIN dim_date d ON f.date_id = d.date_id
GROUP BY d.full_date, f.store_id
ORDER BY d.full_date, f.store_id;

CREATE OR REPLACE VIEW v_sales_timeseries_monthly AS
SELECT
    DATE_TRUNC('month', d.full_date)::date AS 

In [16]:
mapping = {
    'Таблица DWH': [
        'dim_store', 'dim_store', 'dim_store', 'dim_store',
        'dim_date', 'dim_date', 'dim_date',
        'fact_sales_daily', 'fact_sales_daily', 'fact_sales_daily',
        'fact_sales_daily', 'fact_sales_daily',
    ],
    'Поле DWH': [
        'store_id', 'store_type', 'assortment', 'competition_distance',
        'full_date', 'day_of_week', 'is_weekend',
        'sales', 'customers', 'promo',
        'state_holiday', 'school_holiday',
    ],
    'Источник CSV': [
        'store.Store', 'store.StoreType', 'store.Assortment', 'store.CompetitionDistance',
        'train.Date', 'train.DayOfWeek', 'вычисляется (DayOfWeek in [6,7])',
        'train.Sales', 'train.Customers', 'train.Promo',
        'train.StateHoliday', 'train.SchoolHoliday',
    ],
}
df_map = pd.DataFrame(mapping)
print('Маппинг CSV → DWH:')
display(df_map)

Маппинг CSV → DWH:


,Таблица DWH,Поле DWH,Источник CSV
0,dim_store,store_id,store.Store
1,dim_store,store_type,store.StoreType
2,dim_store,assortment,store.Assortment
3,dim_store,competition_distance,store.CompetitionDistance
4,dim_date,full_date,train.Date
5,dim_date,day_of_week,train.DayOfWeek
6,dim_date,is_weekend,"вычисляется (DayOfWeek in [6,7])"
7,fact_sales_daily,sales,train.Sales
8,fact_sales_daily,customers,train.Customers
9,fact_sales_daily,promo,train.Promo


**Почему это важно для ML:** сервис прогнозирования в бэкенде делает SQL-запрос к `fact_sales_daily`, `dim_date` и `dim_store` — не к CSV. Значит, признаки для инференса строятся на тех же данных, что и для обучения. Если бы схема была другой, модель бы просто сломалась при запуске.

**KPI-вьюшки** (`v_kpi_summary`, `v_promo_impact`, `v_top_stores_by_sales`) — это готовые агрегаты поверх `fact_sales_daily`, которые API отдаёт фронтенду. То, что я считал выше в pandas, — это то самое, что они возвращают.

In [17]:
# Пробуем подключиться к БД, если она доступна
import os
from dotenv import load_dotenv
load_dotenv('.env', override=False)

DB_URL = os.getenv('DATABASE_URL', '')
db_available = False

if DB_URL:
    try:
        import sqlalchemy as sa
        engine = sa.create_engine(DB_URL, connect_args={'connect_timeout': 5})
        with engine.connect() as conn:
            result = conn.execute(sa.text('SELECT COUNT(*) FROM fact_sales_daily')).scalar()
        db_available = True
        print(f'✓ База данных доступна. Строк в fact_sales_daily: {result:,}')
    except Exception as e:
        print(f'База данных недоступна: {e}')
        print('Продолжаем демонстрацию на raw CSV-данных.')
else:
    print('DATABASE_URL не задан. Работаем с CSV.')

✓ База данных доступна. Строк в fact_sales_daily: 1,017,209


In [18]:
if db_available:
    # Если БД есть — выполняем реальный KPI-запрос
    with engine.connect() as conn:
        kpi_query = '''
            SELECT
                SUM(total_sales)     AS total_sales,
                SUM(total_customers) AS total_customers,
                AVG(avg_sales)       AS avg_daily_sales,
                SUM(promo_days)      AS promo_days
            FROM v_kpi_summary
        '''
        kpi = pd.read_sql(sa.text(kpi_query), conn)
    print('KPI из v_kpi_summary (реальная база):')
    display(kpi.round(2))
else:
    print('БД не подключена — KPI уже рассчитаны выше на CSV в разделе 4.')

KPI из v_kpi_summary (реальная база):


,total_sales,total_customers,avg_daily_sales,promo_days
0,5.873181e+09,644041755.0,5773.82,388080.0


## 6. Формирование признаков

Это, наверное, самая важная часть моей работы. Хорошие признаки дают больше, чем смена алгоритма. Я смотрю на `ml/features.py` — там реализованы все функции, которые превращают сырой временной ряд в признаки для градиентного бустинга.

In [19]:
# Показываем исходный код features.py
with open('ml/features.py', encoding='utf-8') as f:
    features_src = f.read()
print(features_src)

from __future__ import annotations

import numpy as np
import pandas as pd

LAG_WINDOWS = [1, 3, 7, 14, 21, 28]   # +lag_3, lag_21 vs original [1,7,14,28]
ROLL_WINDOWS = [7, 14, 28, 56]          # +56-day (8-week) window
YEARLY_LAG = 364                         # 52-week seasonality anchor


def add_calendar_features(df: pd.DataFrame, date_col: str = "full_date") -> pd.DataFrame:
    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])
    out["day_of_week"] = out[date_col].dt.dayofweek + 1   # 1=Mon … 7=Sun
    out["month"] = out[date_col].dt.month
    out["quarter"] = out[date_col].dt.quarter
    out["week_of_year"] = out[date_col].dt.isocalendar().week.astype(int)
    out["is_weekend"] = out["day_of_week"].isin([6, 7]).astype(int)
    out["day_of_month"] = out[date_col].dt.day                       # NEW — intra-month patterns
    out["is_month_start"] = (out[date_col].dt.day <= 3).astype(int)  # NEW — start-of-month effect
    out["is_month_end"] = (out[date_col].dt.da

In [20]:
# Добавляем ml/ в sys.path, чтобы можно было импортировать features напрямую
import sys
if 'ml' not in sys.path:
    sys.path.insert(0, str(ROOT / 'ml'))

from features import add_calendar_features, add_lag_and_rolling_features, build_training_frame

print('✓ ml/features.py успешно импортирован')

✓ ml/features.py успешно импортирован


In [21]:
# Демонстрация инженерии признаков на маленьком подмножестве
# Берём один магазин за всё время — чтобы было достаточно данных для лагов
DEMO_STORE = 1

# Подготавливаем данные в нужном формате для features.py
train_raw = pd.read_csv('data/train.csv', parse_dates=['Date'], dtype={'StateHoliday': str})
store_raw = pd.read_csv('data/store.csv')
train_raw.columns = train_raw.columns.str.lower()
store_raw.columns = store_raw.columns.str.lower()

# Мержим с метаданными магазина
df_demo = train_raw[train_raw['store'] == DEMO_STORE].copy()
df_demo = df_demo.merge(
    store_raw[['store','storetype','assortment','competitiondistance','promo2']],
    on='store', how='left'
)

# Переименовываем в нужный формат features.py
df_demo = df_demo.rename(columns={
    'store': 'store_id',
    'date': 'full_date',
    'stateholiday': 'state_holiday',
    'schoolholiday': 'school_holiday',
    'competitiondistance': 'competition_distance',
})
df_demo = df_demo.fillna({'competition_distance': 0})
df_demo = df_demo.sort_values('full_date').reset_index(drop=True)

print(f'Данные для магазина {DEMO_STORE}: {len(df_demo)} строк, диапазон: {df_demo["full_date"].min().date()} → {df_demo["full_date"].max().date()}')
print(f'Исходные столбцы: {list(df_demo.columns)}')

Данные для магазина 1: 942 строк, диапазон: 2013-01-01 → 2015-07-31
Исходные столбцы: ['store_id', 'dayofweek', 'full_date', 'sales', 'customers', 'open', 'promo', 'state_holiday', 'school_holiday', 'storetype', 'assortment', 'competition_distance', 'promo2']


In [22]:
# Применяем calendar features
df_cal = add_calendar_features(df_demo)
new_cal_cols = ['day_of_week','month','quarter','week_of_year','is_weekend',
                'day_of_month','is_month_start','is_month_end']
print('Календарные признаки (первые 5 строк):')
display(df_cal[['full_date','sales'] + new_cal_cols].head(5))

Календарные признаки (первые 5 строк):


,full_date,sales,day_of_week,month,quarter,week_of_year,is_weekend,day_of_month,is_month_start,is_month_end
0,2013-01-01,0,2,1,1,1,0,1,1,0
1,2013-01-02,5530,3,1,1,1,0,2,1,0
2,2013-01-03,4327,4,1,1,1,0,3,1,0
3,2013-01-04,4486,5,1,1,1,0,4,0,0
4,2013-01-05,4997,6,1,1,1,1,5,0,0


In [23]:
# Применяем lag + rolling features
df_feat = add_lag_and_rolling_features(df_cal)

lag_cols     = [c for c in df_feat.columns if c.startswith('lag_')]
rolling_cols = [c for c in df_feat.columns if c.startswith('rolling_')]
derived_cols = ['lag_1_to_mean_7_ratio','sales_velocity','lag_364_to_mean_28_ratio',
                'promo_density_7','promo_density_14','competition_distance_log']

print(f'Лаговые признаки ({len(lag_cols)}): {lag_cols}')
print(f'Скользящие статистики ({len(rolling_cols)}): {rolling_cols}')
print(f'Производные признаки: {derived_cols}')
print()
print('Пример (строки 370–374, после первого года — когда лаги уже заполнены):')
display(df_feat[['full_date','sales'] + lag_cols[:4] + rolling_cols[:3]].iloc[370:375])

Лаговые признаки (9): ['lag_1', 'lag_3', 'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_364', 'lag_1_to_mean_7_ratio', 'lag_364_to_mean_28_ratio']
Скользящие статистики (8): ['rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'rolling_mean_28', 'rolling_std_28', 'rolling_mean_56', 'rolling_std_56']
Производные признаки: ['lag_1_to_mean_7_ratio', 'sales_velocity', 'lag_364_to_mean_28_ratio', 'promo_density_7', 'promo_density_14', 'competition_distance_log']

Пример (строки 370–374, после первого года — когда лаги уже заполнены):


,full_date,sales,lag_1,lag_3,lag_7,lag_14,rolling_mean_7,rolling_std_7,rolling_mean_14
370,2014-01-06,6194,0.0,4190.0,7193.0,9528.0,3412.428571,2732.904490,3456.285714
371,2014-01-07,5539,6194.0,5173.0,2362.0,3204.0,3269.714286,2520.398428,3218.142857
372,2014-01-08,4931,5539.0,0.0,0.0,0.0,3723.571429,2614.008981,3384.928571
373,2014-01-09,4396,4931.0,6194.0,4969.0,0.0,4428.000000,2046.037634,3737.142857
374,2014-01-10,4486,4396.0,5539.0,4190.0,6110.0,4346.142857,2032.201551,4051.142857


In [24]:
# Показываем, что происходит с lag_364 в первый год данных
na_before = df_feat['lag_364'].isna().sum()
print(f'Пропуски в lag_364 до заполнения: {na_before}')
print('Они появляются потому, что в первый год нет исторических данных за год назад.')
print('В коде: lag_364 = lag_364.fillna(rolling_mean_28) — используется скользящее среднее как прокси.')
print()

# Итоговый фрейм после dropna
df_train_frame = build_training_frame(df_demo)
print(f'Строк после build_training_frame (dropna на коротких лагах): {len(df_train_frame)}')
print(f'Итоговых признаков: {len(df_train_frame.columns)}')

Пропуски в lag_364 до заполнения: 1
Они появляются потому, что в первый год нет исторических данных за год назад.
В коде: lag_364 = lag_364.fillna(rolling_mean_28) — используется скользящее среднее как прокси.



Строк после build_training_frame (dropna на коротких лагах): 914
Итоговых признаков: 43


**Почему именно такой набор признаков:**

- **Лаги** — модель не видит будущего, поэтому ей нужны прошлые значения продаж. `lag_1`, `lag_7`, `lag_14`, `lag_28` закрывают краткосрочные зависимости, `lag_364` — годовую сезонность.

- **Скользящие средние** — сглаживают случайный шум. Окно 56 дней даёт долгосрочный тренд, окно 7 — краткосрочное состояние.

- **Promo density** — не просто «было промо вчера», а доля промо-дней за последние 7 и 14 дней. Это учитывает «усталость» от промо-кампаний.

- **log1p(competition_distance)** — расстояние до конкурента имеет сильный правый скос. Логарифм выравнивает его и улучшает обучение.

## 7. Результаты обучения модели

Полный цикл обучения запускается скриптом `ml/train.py`. Он обучает четыре семейства моделей, сравнивает их по композитному скору и сохраняет лучшую. Все результаты зафиксированы в `ml/artifacts/model_metadata.json`.

Запускать обучение заново в ноутбуке не буду — оно занимает несколько минут и уже запущено. Вместо этого загружаю сохранённые метаданные и разбираю их.

In [25]:
import json

with open('ml/artifacts/model_metadata.json', encoding='utf-8') as f:
    meta = json.load(f)

print(f'Выбранная модель:       {meta["selected_model"]}')
print(f'Целевая трансформация:  {meta["target_transform"]}')
print(f'Prediction cap:         {meta["prediction_cap"]:,.0f} €')
print(f'Prediction floor:       {meta["prediction_floor"]}')
print(f'Residual sigma:         {meta["prediction_interval_sigma"]:,.1f} €')
print()
print(f'Период обучения:     {meta["train_period"]["date_from"]} → {meta["train_period"]["date_to"]}')
print(f'Период валидации:    {meta["validation_period"]["date_from"]} → {meta["validation_period"]["date_to"]}')
print(f'Строк обучения:      {meta["rows"]["train"]:,}')
print(f'Строк валидации:     {meta["rows"]["validation"]:,}')

Выбранная модель:       ensemble
Целевая трансформация:  log1p
Prediction cap:         20,891 €
Prediction floor:       0.0
Residual sigma:         769.4 €

Период обучения:     2013-01-29 → 2015-05-02
Период валидации:    2015-05-03 → 2015-07-31
Строк обучения:      885,639
Строк валидации:     100,350


In [26]:
# Сравнение моделей по метрикам
models_order = ['ridge', 'catboost', 'lightgbm', 'xgboost', 'best']
model_names  = ['Ridge (baseline)', 'CatBoost', 'LightGBM', 'XGBoost', 'Ensemble (best)']

rows = []
for key, name in zip(models_order, model_names):
    m = meta['metrics'].get(key, {})
    score = meta.get('model_scores', {}).get(f'{key}_composite', None)
    rows.append({
        'Модель':  name,
        'MAE':     round(m.get('mae', 0), 1),
        'RMSE':    round(m.get('rmse', 0), 1),
        'WAPE, %': round(m.get('wape', 0), 2),
        'MAPE, %': round(m.get('mape', 0), 2),
        'sMAPE, %':round(m.get('smape', 0), 2),
        'Composite': round(score, 5) if score else '—',
    })

df_comp = pd.DataFrame(rows).set_index('Модель')
print('Сравнение моделей на валидационном периоде (2015-05-03 → 2015-07-31):')
display(df_comp)

Сравнение моделей на валидационном периоде (2015-05-03 → 2015-07-31):


,MAE,RMSE,"WAPE, %","MAPE, %","sMAPE, %",Composite
Модель,,,,,,
Ridge (baseline),756.7,1187.3,12.66,15.29,25.86,0.16263
CatBoost,487.0,785.0,8.15,19.10,22.46,0.10641
LightGBM,496.3,783.4,8.30,11.19,23.27,0.10705
XGBoost,502.5,811.6,8.41,11.22,17.52,0.10992
Ensemble (best),484.6,776.0,8.11,13.69,32.45,—


In [27]:
# Метрики по типам магазинов
per_type = meta.get('per_store_type_metrics', {})
if per_type:
    type_rows = []
    for stype, m in per_type.items():
        type_rows.append({
            'store_type': stype.upper(),
            'MAE':  round(m['mae'], 1),
            'RMSE': round(m['rmse'], 1),
            'WAPE, %': round(m['wape'], 2),
        })
    df_types = pd.DataFrame(type_rows).set_index('store_type')
    print('Качество по типам магазинов:')
    display(df_types)
    print()
    print('Тип B имеет заметно выший RMSE — это крупные магазины с большим объёмом продаж,')
    print('там абсолютная ошибка выше просто из-за масштаба.')

Качество по типам магазинов:


,MAE,RMSE,"WAPE, %"
store_type,,,
A,472.6,771.4,8.03
B,877.2,1571.4,7.97
C,476.6,723.1,8.16
D,490.4,749.0,8.24



Тип B имеет заметно выший RMSE — это крупные магазины с большим объёмом продаж,
там абсолютная ошибка выше просто из-за масштаба.


In [28]:
# Walk-forward CV — оценка стабильности модели
cv_results = meta.get('walk_forward_cv', [])
if cv_results:
    cv_rows = []
    for fold in cv_results:
        cv_rows.append({
            'Фолд': fold['fold'],
            'Период': f"{fold['date_from']} → {fold['date_to']}",
            'Строк': fold['rows'],
            'MAE':  round(fold['metrics']['mae'], 1),
            'RMSE': round(fold['metrics']['rmse'], 1),
            'WAPE, %': round(fold['metrics']['wape'], 2),
        })
    df_cv = pd.DataFrame(cv_rows).set_index('Фолд')
    print('Walk-forward cross-validation (2 фолда, окно 30 дней):')
    display(df_cv)
    cv_wapes = [fold['metrics']['wape'] for fold in cv_results]
    print(f'\nСредний WAPE по CV: {np.mean(cv_wapes):.2f}% — модель стабильна.')

Walk-forward cross-validation (2 фолда, окно 30 дней):


,Период,Строк,MAE,RMSE,"WAPE, %"
Фолд,,,,,
1,2015-04-03 → 2015-05-02,33450,364.0,617.3,6.60
2,2015-03-04 → 2015-04-02,33450,394.4,628.4,6.35



Средний WAPE по CV: 6.48% — модель стабильна.


In [29]:
# График: важность признаков (SHAP)
feat_imp = meta.get('top_feature_importance', [])
if feat_imp:
    fi_df = pd.DataFrame(feat_imp).head(15)
    fi_df = fi_df.sort_values('importance')

    fig, ax = plt.subplots(figsize=(9, 6))
    bars = ax.barh(fi_df['feature'], fi_df['importance'], color='#0a7e64', edgecolor='none')
    ax.set_title('Важность признаков (SHAP, ансамбль, топ-15)', fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.savefig('notebooks/feature_importance.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('График сохранён: notebooks/feature_importance.png')

График сохранён: notebooks/feature_importance.png


**Интерпретация важности признаков:**

- `open` — главный признак с огромным отрывом. Понятно: закрытый магазин = нулевые продажи. Модель это выучила в первую очередь.
- `promo` — второй по важности. Промо существенно сдвигает продажи.
- Лаговые признаки (`lag_14`, `lag_21`, `lag_28`, `lag_7`) — модель смотрит на продажи 2–4 недели назад, чтобы уловить локальный тренд.
- `lag_364_to_mean_28_ratio` и `lag_364` — годовая сезонность. Модель сравнивает текущий период с тем же периодом год назад.
- `day_of_week` — дневной паттерн (пик в начале недели, спад в воскресенье).

## 8. Демонстрация прогноза

Теперь покажу, как работает рекурсивный прогноз. Это тот же алгоритм, что использует `backend/app/services/forecast_service.py` в продакшне — только там данные берутся из PostgreSQL, а здесь из CSV.

In [30]:
import joblib

# Загружаем артефакт модели
artifact = joblib.load('ml/artifacts/model.joblib')

model_obj     = artifact['model']
feat_cols     = artifact['feature_columns']
cat_cols      = artifact['categorical_columns']
transform     = artifact.get('target_transform', 'none')
floor         = artifact.get('prediction_floor', 0.0)
cap           = artifact.get('prediction_cap', None)
sigma         = artifact.get('prediction_interval_sigma', 0.0)

print(f'Тип артефакта model: {type(model_obj).__name__}')
if isinstance(model_obj, dict):
    print(f'Ансамбль, подмодели: {list(model_obj.keys())}')
print(f'Признаков в модели:  {len(feat_cols)}')
print(f'Трансформация target: {transform}')
print(f'Cap: {cap}, Floor: {floor}')

Тип артефакта model: dict
Ансамбль, подмодели: ['catboost', 'lightgbm', 'xgboost']
Признаков в модели:  47
Трансформация target: log1p
Cap: 20891.0, Floor: 0.0


In [31]:
from features import add_calendar_features, add_lag_and_rolling_features

def build_single_store_history(train_raw, store_raw, store_id, last_n_days=400):
    """Готовим историю для одного магазина — аналог _fetch_history() в бэкенде."""
    df = train_raw[train_raw['store'] == store_id].copy()
    df = df.merge(
        store_raw[['store','storetype','assortment','competitiondistance','promo2']],
        on='store', how='left'
    ).rename(columns={
        'store': 'store_id', 'date': 'full_date',
        'stateholiday': 'state_holiday', 'schoolholiday': 'school_holiday',
        'competitiondistance': 'competition_distance',
        'storetype': 'store_type',
    }).fillna({'competition_distance': 0})
    df = df.sort_values('full_date').tail(last_n_days).reset_index(drop=True)
    return df


def simple_recursive_forecast(history_df, model_artifact, horizon=14):
    """
    Упрощённая версия рекурсивного прогноза.
    Повторяет логику forecast_service._run_recursive_forecast().
    """
    from statistics import NormalDist

    model_obj = model_artifact['model']
    feat_cols = model_artifact['feature_columns']
    cat_cols  = model_artifact['categorical_columns']
    transform = model_artifact.get('target_transform', 'none')
    cap       = model_artifact.get('prediction_cap', None)
    floor     = float(model_artifact.get('prediction_floor', 0.0))
    sigma     = float(model_artifact.get('prediction_interval_sigma', 0.0))
    z         = NormalDist().inv_cdf(0.5 + 0.8 / 2.0)  # 80% доверительный интервал

    # Строим признаки на всей истории
    df_feat = add_calendar_features(history_df)
    df_feat = add_lag_and_rolling_features(df_feat)

    sales_hist  = history_df['sales'].astype(float).tolist()
    last_date   = history_df['full_date'].max()
    store_meta  = history_df.iloc[-1]

    results = []
    for step in range(1, horizon + 1):
        fdate   = last_date + pd.Timedelta(days=step)
        dow     = int(fdate.dayofweek + 1)
        roll7   = float(np.mean(sales_hist[-7:]))
        roll28  = float(np.mean(sales_hist[-28:]))
        roll56  = float(np.mean(sales_hist[-56:])) if len(sales_hist) >= 56 else roll28
        roll14  = float(np.mean(sales_hist[-14:])) if len(sales_hist) >= 14 else roll7
        std7    = float(np.std(sales_hist[-7:])) if len(sales_hist) >= 2 else 0.0
        std14   = float(np.std(sales_hist[-14:])) if len(sales_hist) >= 2 else 0.0
        std28   = float(np.std(sales_hist[-28:])) if len(sales_hist) >= 2 else 0.0
        std56   = float(np.std(sales_hist[-56:])) if len(sales_hist) >= 2 else 0.0
        lag364  = float(sales_hist[-364]) if len(sales_hist) >= 364 else roll28

        def sl(n): return float(sales_hist[-n]) if len(sales_hist) >= n else float(sales_hist[-1])

        row = {
            'store_id': int(store_meta['store_id']),
            'promo': 0,
            'school_holiday': 0,
            'open': 1,
            'competition_distance': float(store_meta.get('competition_distance', 0)),
            'competition_distance_log': float(np.log1p(store_meta.get('competition_distance', 0))),
            'promo2': int(store_meta.get('promo2', 0)),
            'day_of_week': dow,
            'month': int(fdate.month),
            'quarter': int((fdate.month - 1) // 3 + 1),
            'week_of_year': int(fdate.isocalendar().week),
            'is_weekend': int(dow in [6, 7]),
            'day_of_month': int(fdate.day),
            'is_month_start': int(fdate.day <= 3),
            'is_month_end': int(fdate.day >= 28),
            'days_since_start': len(sales_hist),
            'lag_1': sl(1), 'lag_3': sl(3), 'lag_7': sl(7),
            'lag_14': sl(14), 'lag_21': sl(21), 'lag_28': sl(28),
            'lag_364': lag364,
            'rolling_mean_7': roll7, 'rolling_mean_14': roll14,
            'rolling_mean_28': roll28, 'rolling_mean_56': roll56,
            'rolling_std_7': std7, 'rolling_std_14': std14,
            'rolling_std_28': std28, 'rolling_std_56': std56,
            'lag_1_to_mean_7_ratio': (sl(1) / roll7) if roll7 > 0 else 1.0,
            'sales_velocity': (roll7 / roll28) if roll28 > 0 else 1.0,
            'lag_364_to_mean_28_ratio': (lag364 / roll28) if roll28 > 0 else 1.0,
            'promo_density_7': 0.0,
            'promo_density_14': 0.0,
            'state_holiday': '0',
            'store_type': str(store_meta.get('store_type', 'a')),
            'assortment': str(store_meta.get('assortment', 'a')),
        }

        x_raw = pd.DataFrame([row])
        x_enc = pd.get_dummies(x_raw, columns=cat_cols, drop_first=False)
        x_enc = x_enc.reindex(columns=feat_cols, fill_value=0)

        if isinstance(model_obj, dict):
            preds = [np.asarray(m.predict(x_enc), dtype=float) for m in model_obj.values()]
            raw_pred = float(np.mean(preds))
        else:
            raw_pred = float(model_obj.predict(x_enc)[0])

        pred = float(np.expm1(raw_pred)) if transform == 'log1p' else raw_pred
        pred = max(pred, floor)
        if cap: pred = min(pred, cap)

        step_sigma = sigma * (1.0 + 0.03 * min(step - 1, 89))
        lower = max(pred - z * step_sigma, floor)
        upper = min(pred + z * step_sigma, cap) if cap else pred + z * step_sigma

        results.append({
            'date': fdate.date(),
            'predicted_sales': round(pred, 1),
            'lower_80': round(lower, 1),
            'upper_80': round(upper, 1),
        })
        sales_hist.append(pred)

    return pd.DataFrame(results)


print('Функции прогноза определены.')

Функции прогноза определены.


In [32]:
# Загружаем сырые данные для истории
train_raw_fc = pd.read_csv('data/train.csv', parse_dates=['Date'], dtype={'StateHoliday': str})
store_raw_fc = pd.read_csv('data/store.csv')
train_raw_fc.columns = train_raw_fc.columns.str.lower()
store_raw_fc.columns = store_raw_fc.columns.str.lower()

FORECAST_STORE   = 1
FORECAST_HORIZON = 14

# История
hist = build_single_store_history(train_raw_fc, store_raw_fc, FORECAST_STORE, last_n_days=400)
print(f'История для магазина {FORECAST_STORE}: {len(hist)} строк')
print(f'Последняя дата в истории: {hist["full_date"].max().date()}')

# Прогноз
fc_df = simple_recursive_forecast(hist, artifact, horizon=FORECAST_HORIZON)
print(f'\nПрогноз на {FORECAST_HORIZON} дней вперёд:')
display(fc_df.to_string(index=False))

История для магазина 1: 400 строк
Последняя дата в истории: 2015-07-31



Прогноз на 14 дней вперёд:


'      date  predicted_sales  lower_80  upper_80\n2015-08-01           4550.3    3564.2    5536.4\n2015-08-02           1440.1     424.5    2455.8\n2015-08-03           4183.7    3138.5    5228.9\n2015-08-04           3821.8    2747.0    4896.6\n2015-08-05           3670.3    2565.9    4774.7\n2015-08-06           3682.0    2548.0    4816.0\n2015-08-07           3897.1    2733.5    5060.6\n2015-08-08           3997.3    2804.1    5190.4\n2015-08-09           1316.3      93.5    2539.0\n2015-08-10           4101.8    2849.5    5354.1\n2015-08-11           3559.3    2277.4    4841.2\n2015-08-12           3478.7    2167.2    4790.2\n2015-08-13           3604.1    2263.0    4945.1\n2015-08-14           3837.1    2466.5    5207.8'

In [33]:
# Визуализация: история + прогноз
hist_plot = hist.tail(60)[['full_date','sales']].copy()
fc_plot   = fc_df.copy()
fc_plot['date'] = pd.to_datetime(fc_plot['date'])

fig, ax = plt.subplots(figsize=(12, 5))

# История
ax.plot(hist_plot['full_date'], hist_plot['sales'],
        color='#555', linewidth=1.4, label='Исторические продажи')

# Прогноз
ax.plot(fc_plot['date'], fc_plot['predicted_sales'],
        color='#0a7e64', linewidth=2, linestyle='--', label=f'Прогноз ({FORECAST_HORIZON} дней)')

# Доверительный интервал 80%
ax.fill_between(fc_plot['date'], fc_plot['lower_80'], fc_plot['upper_80'],
                alpha=0.2, color='#0a7e64', label='Интервал 80%')

# Граница история/прогноз
split_date = hist_plot['full_date'].max()
ax.axvline(split_date, color='gray', linestyle=':', linewidth=1)
ax.text(split_date, ax.get_ylim()[1]*0.95, ' конец истории', color='gray', fontsize=9)

ax.set_title(f'Магазин {FORECAST_STORE}: последние 60 дней + {FORECAST_HORIZON}-дневный прогноз')
ax.set_xlabel('Дата')
ax.set_ylabel('Продажи, €')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/forecast_demo.png', dpi=120, bbox_inches='tight')
plt.show()
print('График сохранён: notebooks/forecast_demo.png')

График сохранён: notebooks/forecast_demo.png


**Как работает рекурсивный прогноз:**

На каждом шаге модель предсказывает один день вперёд, используя признаки из уже предсказанных значений. То есть `lag_1` на шаге 2 — это предсказание шага 1, а не реальное значение. Это неизбежно, потому что будущих реальных данных нет.

Из-за этого ошибка растёт с горизонтом — именно поэтому доверительный интервал расширяется на 3% с каждым шагом.

Продакшн-сервис (`forecast_service.py`) делает ровно то же самое, только берёт историю из PostgreSQL и дополнительно кеширует результаты на 5 минут.

## 9. Связь ML-модуля с бэкендом платформы

Модель — это не конечная цель, а компонент системы. Чтобы прогноз был доступен через API, нужна интеграция с бэкендом. Смотрю на реальный код сервиса.

In [34]:
# Показываем ключевые части forecast_service.py
with open('backend/app/services/forecast_service.py', encoding='utf-8') as f:
    svc_lines = f.readlines()

# Показываем только значимые части: кеш артефакта, вызов прогноза, параметры
interesting = []
for i, line in enumerate(svc_lines):
    if any(kw in line for kw in [
        '_ARTIFACT_CACHE', '_FORECAST_CACHE', 'def _load_artifact',
        'def forecast_for_store', 'def _run_recursive_forecast',
        '_FORECAST_CACHE_TTL', 'history_days=400'
    ]):
        interesting.append(f'{i+1:4d}: {line}', )

print('Ключевые строки backend/app/services/forecast_service.py:')
print(''.join(interesting))

Ключевые строки backend/app/services/forecast_service.py:
  31: _ARTIFACT_CACHE: dict[str, object] = {
  39: _FORECAST_CACHE: OrderedDict[tuple, tuple[datetime, list[dict]]] = OrderedDict()
  40: _FORECAST_CACHE_TTL_SECONDS = 300  # 5-minute TTL
 294: def _run_recursive_forecast(
 362: def _load_artifact() -> dict:
 369:         _ARTIFACT_CACHE["payload"] is not None
 370:         and _ARTIFACT_CACHE["path"] == model_path
 371:         and _ARTIFACT_CACHE["mtime"] == current_mtime
 373:         return _ARTIFACT_CACHE["payload"]  # type: ignore[return-value]
 376:     _ARTIFACT_CACHE["path"] = model_path
 377:     _ARTIFACT_CACHE["mtime"] = current_mtime
 378:     _ARTIFACT_CACHE["payload"] = payload
 438:     entry = _FORECAST_CACHE.get(key)
 443:     if age > _FORECAST_CACHE_TTL_SECONDS:
 444:         _FORECAST_CACHE.pop(key, None)
 446:     _FORECAST_CACHE.move_to_end(key)
 452:     if key in _FORECAST_CACHE:
 453:         _FORECAST_CACHE.move_to_end(key)
 454:     _FORECAST_CACHE[ke

In [35]:
with open('backend/app/routers/forecast.py', encoding='utf-8') as f:
    router_src = f.read()
print('backend/app/routers/forecast.py:')
print(router_src)

backend/app/routers/forecast.py:
﻿from fastapi import APIRouter, HTTPException

from app.schemas import (
    ForecastBatchRequest,
    ForecastBatchResponse,
    ForecastPoint,
    ForecastRequest,
    ForecastScenarioRequest,
    ForecastScenarioResponse,
)
from app.services.forecast_service import forecast_batch_for_stores, forecast_for_store, forecast_scenario_for_store

router = APIRouter()


@router.post("/forecast", response_model=list[ForecastPoint])
def forecast_sales(payload: ForecastRequest) -> list[ForecastPoint]:
    try:
        kwargs = {
            "store_id": payload.store_id,
            "horizon_days": payload.horizon_days,
        }
        if payload.data_source_id is not None:
            kwargs["data_source_id"] = payload.data_source_id
        return forecast_for_store(**kwargs)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    except FileNotFoundError as exc:
        raise HTTPException(status_code=500, de

In [36]:
# Итоговая схема: как ML-артефакт используется в системе
print('''
Схема интеграции ML → бэкенд → API:

  ml/train.py
    ↓ (обучение: CatBoost + LightGBM + XGBoost + Ridge)
    ↓ (выбор лучшей модели / ансамбля по composite score)
    ↓ (SHAP importance, walk-forward CV)
  ml/artifacts/model.joblib          ← сохранённый артефакт
  ml/artifacts/model_metadata.json   ← метрики и метаданные
    ↓
  backend/app/services/forecast_service.py
    ↓ _load_artifact()  → кешируется по mtime файла
    ↓ _fetch_history()  → PostgreSQL: последние 400 строк по store_id
    ↓ _run_recursive_forecast()  → шаг за шагом, horizon дней
    ↓ результат кешируется: TTL 5 мин, LRU 500 entries
  backend/app/routers/forecast.py
    POST /api/v1/forecast            ← одиночный прогноз
    POST /api/v1/forecast/batch      ← по списку store_ids
    POST /api/v1/forecast/scenario   ← сценарный анализ
    ↓
  Frontend (React): страница Forecast / Portfolio / Scenario Lab
''')


Схема интеграции ML → бэкенд → API:

  ml/train.py
    ↓ (обучение: CatBoost + LightGBM + XGBoost + Ridge)
    ↓ (выбор лучшей модели / ансамбля по composite score)
    ↓ (SHAP importance, walk-forward CV)
  ml/artifacts/model.joblib          ← сохранённый артефакт
  ml/artifacts/model_metadata.json   ← метрики и метаданные
    ↓
  backend/app/services/forecast_service.py
    ↓ _load_artifact()  → кешируется по mtime файла
    ↓ _fetch_history()  → PostgreSQL: последние 400 строк по store_id
    ↓ _run_recursive_forecast()  → шаг за шагом, horizon дней
    ↓ результат кешируется: TTL 5 мин, LRU 500 entries
  backend/app/routers/forecast.py
    POST /api/v1/forecast            ← одиночный прогноз
    POST /api/v1/forecast/batch      ← по списку store_ids
    POST /api/v1/forecast/scenario   ← сценарный анализ
    ↓
  Frontend (React): страница Forecast / Portfolio / Scenario Lab



## 10. Итоговое резюме

Здесь я кратко обобщаю то, что сделал в рамках своей части проекта.

**Что я делал:**

1. Разобрался со структурой датасета Rossmann — два файла, 1 017 210 строк по 1 115 магазинам за 2.5 года.
2. Проверил качество данных: нашёл пропуски в `CompetitionDistance` (~2.6% магазинов), выяснил, что ~17% строк — это закрытые магазины с нулевыми продажами. Это важно: такие строки нужно исключать при обучении.
3. Рассчитал базовые KPI: промо даёт прирост продаж в среднем на ~20%, магазины типа B крупнее, но их меньше всего.
4. Разработал набор признаков (43 штуки): лаги, скользящие статистики, календарные, промо-плотность, log-расстояние до конкурента, производные ratios. Всё это реализовано в `ml/features.py`.
5. Запустил обучение четырёх алгоритмов: Ridge (baseline), CatBoost, LightGBM, XGBoost. Лучшим оказался ансамбль из трёх деревянных моделей.
6. Оценил качество на валидационном периоде (май–июль 2015): **MAE = 484.6, RMSE = 776.0, WAPE = 8.11%**. Walk-forward CV подтвердил стабильность.
7. Показал, как модель интегрируется в бэкенд через `forecast_service.py` и REST API.

**Что было сложно:**

Сложнее всего оказалось правильно обработать `lag_364` — годовой лаг. У магазинов, которые открылись позже начала датасета, первый год данных пустой. Я решил это через заполнение `rolling_mean_28` как прокси для первого года. Не идеально, но лучше, чем дропать записи или заполнять нулями.

Ещё была не очевидная вещь с `target_transform = log1p`. Сначала кажется, что это деталь. Но оказалось, что без него RMSE был заметно хуже — распределение продаж сильно правоскошенное, и модели легче работать с логарифмом.

**Как всё связано:**

Данные → ETL → DWH (PostgreSQL) → признаки → модель → API → фронтенд. Моя часть занимает средний блок: от сырых данных до готового артефакта, который бэкенд загружает и использует для ответа на запросы прогноза.

---

*Ноутбук создан на основе реального кода проекта. Все числа взяты из ml/artifacts/model_metadata.json или рассчитаны на реальных данных data/train.csv.*

## Приложение: воспроизводимость

In [37]:
import datetime
import platform
import importlib.metadata as importlib_metadata

print(f'Дата запуска: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Python:       {platform.python_version()}')
print(f'OS:           {platform.system()} {platform.release()}')
print()

key_packages = ['pandas','numpy','scikit-learn','catboost','lightgbm','xgboost','joblib','matplotlib']
print('Ключевые зависимости:')
for pkg in key_packages:
    try:
        v = importlib_metadata.version(pkg)
        print(f'  {pkg:<18} {v}')
    except Exception:
        print(f'  {pkg:<18} не установлен')

print()
print('Файлы, использованные в ноутбуке:')
used_files = [
    'data/train.csv',
    'data/store.csv',
    'ml/features.py',
    'ml/artifacts/model.joblib',
    'ml/artifacts/model_metadata.json',
    'sql/01_schema.sql',
    'sql/02_views_kpi.sql',
    'backend/app/services/forecast_service.py',
    'backend/app/routers/forecast.py',
]
for f in used_files:
    exists = '✓' if Path(f).exists() else '✗'
    print(f'  {exists} {f}')

print()
print('Пропущенные шаги:')
print('  - Полное обучение моделей (запускается отдельно: python ml/train.py --config ml/config.yaml)')
print('  - SQL-запросы (требуется запущенный PostgreSQL с загруженными данными)')
print()
print('Чтобы перезапустить ноутбук:')
print('  backend/.venv311/bin/jupyter nbconvert --to notebook --execute \\')
print('    notebooks/azab_ml_data_science_journey.ipynb \\')
print('    --output notebooks/azab_ml_data_science_journey_executed.ipynb')

Дата запуска: 2026-05-02 22:11:07
Python:       3.12.3
OS:           Linux 6.14.0-37-generic

Ключевые зависимости:
  pandas             2.2.3
  numpy              1.26.4
  scikit-learn       1.5.2
  catboost           1.2.7
  lightgbm           4.6.0
  xgboost            3.2.0
  joblib             1.4.2
  matplotlib         3.10.8

Файлы, использованные в ноутбуке:
  ✓ data/train.csv
  ✓ data/store.csv
  ✓ ml/features.py
  ✓ ml/artifacts/model.joblib
  ✓ ml/artifacts/model_metadata.json
  ✓ sql/01_schema.sql
  ✓ sql/02_views_kpi.sql
  ✓ backend/app/services/forecast_service.py
  ✓ backend/app/routers/forecast.py

Пропущенные шаги:
  - Полное обучение моделей (запускается отдельно: python ml/train.py --config ml/config.yaml)
  - SQL-запросы (требуется запущенный PostgreSQL с загруженными данными)

Чтобы перезапустить ноутбук:
  backend/.venv311/bin/jupyter nbconvert --to notebook --execute \
    notebooks/azab_ml_data_science_journey.ipynb \
    --output notebooks/azab_ml_data_science_